# 0.0.Download loaddata csv from pinned pediatric atlas pilot repo commit
This notebook downloads the CellProfiler loaddata from pinned pediatric atlas pilot repo commit.

The downloaded loaddata will serve later in this analysis directory as the image file index. 

In [1]:
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import urlopen

from utils.resolve_root import resolve_root
from utils.download_csv import download_csv_files

## Download pathing

In [2]:
metadata_download_path = resolve_root(globals().get("__file__")) / "metadata"
metadata_download_path.mkdir(parents=True, exist_ok=True)

## Download loaddata to use as file index

In [3]:
source_repo = "https://github.com/WayScience/pediatric_cancer_atlas_profiling.git"
pin_commit = "6129e23e64cad5402c6d1d5ba679ff5f3eda82e4"

# Save this file directly under metadata/.
barcode = "0.download_data/metadata/platemaps/Barcode_platemap_pilot_data.csv"

# Save these files under metadata/platemaps/.
platemaps = [
    "0.download_data/metadata/platemaps/Assay_Plate1_platemap.csv",
    "0.download_data/metadata/platemaps/Assay_Plate2_platemap.csv",
]

# Save these files under metadata/loaddatas/.
loaddatas = [
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143976_concatenated.csv", # 24 hrs
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143977_concatenated.csv", # 24 hrs
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143978_concatenated.csv", # 48 hrs
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143979_concatenated.csv", # 48 hrs
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143980_concatenated.csv", # 72 hrs
    "1.illumination_correction/loaddata_csvs/Round_1_data/BR00143981_concatenated.csv", # 72 hrs
]

In [4]:
csv_downloads = {
    metadata_download_path: [barcode],
    metadata_download_path / "platemaps": platemaps,
    metadata_download_path / "loaddatas": loaddatas,
}

downloaded_files = []

for destination_dir, source_paths in csv_downloads.items():
    downloaded_files.extend(
        download_csv_files(
            source_paths=source_paths,
            destination_dir=destination_dir,
            repo_url=source_repo,
            commit=pin_commit,
        )
    )

print(f"Downloaded {len(downloaded_files)} CSV files.")

Downloaded 0.download_data/metadata/platemaps/Barcode_platemap_pilot_data.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/Barcode_platemap_pilot_data.csv
Downloaded 0.download_data/metadata/platemaps/Assay_Plate1_platemap.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/platemaps/Assay_Plate1_platemap.csv
Downloaded 0.download_data/metadata/platemaps/Assay_Plate2_platemap.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/platemaps/Assay_Plate2_platemap.csv
Downloaded 1.illumination_correction/loaddata_csvs/Round_1_data/BR00143976_concatenated.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143976_concatenated.csv
Downloaded 1.illumination_correction/loaddata_csvs/Round_1_data/BR00143977_concatenated.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143977_concatenated.csv
Downloaded 1.illumination_correction/loaddata_csvs/Round_1_data/BR00143978

## Download whole-image quality control
Download each plate's QC image table from the same pinned commit, saving it under `metadata/whole_img_qc_output/<plate_id>.csv`. Missing files produce a warning and are skipped.

In [5]:
qc_destination_dir = metadata_download_path / "whole_img_qc_output"
qc_destination_dir.mkdir(parents=True, exist_ok=True)
qc_source_dir = "1.illumination_correction/img_quality_control/whole_img_qc_output"
repository_slug = source_repo.removesuffix(".git").removeprefix("https://github.com/")
qc_downloaded_files = []

for plate_number in range(143976, 143982):
    plate_id = f"BR{plate_number:08d}"
    source_path = f"{qc_source_dir}/{plate_id}/Image.csv"
    raw_url = (
        f"https://raw.githubusercontent.com/{repository_slug}/{pin_commit}/"
        f"{quote(source_path, safe='/')}"
    )
    destination_path = qc_destination_dir / f"{plate_id}.csv"

    try:
        # Read the full response before replacing any existing local file.
        with urlopen(raw_url, timeout=60) as response:
            file_contents = response.read()
    except HTTPError as error:
        if error.code != 404:
            raise
        print(f"WARNING: QC file does not exist at pinned commit; skipping {source_path}")
        continue

    destination_path.write_bytes(file_contents)
    qc_downloaded_files.append(destination_path)
    print(f"Downloaded {source_path} -> {destination_path}")

print(f"Downloaded {len(qc_downloaded_files)} of 6 anticipated QC CSV files.")

Downloaded 1.illumination_correction/img_quality_control/whole_img_qc_output/BR00143976/Image.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/whole_img_qc_output/BR00143976.csv
Downloaded 1.illumination_correction/img_quality_control/whole_img_qc_output/BR00143977/Image.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/whole_img_qc_output/BR00143977.csv
Downloaded 1.illumination_correction/img_quality_control/whole_img_qc_output/BR00143978/Image.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/whole_img_qc_output/BR00143978.csv
Downloaded 1.illumination_correction/img_quality_control/whole_img_qc_output/BR00143979/Image.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/whole_img_qc_output/BR00143979.csv
Downloaded 1.illumination_correction/img_quality_control/whole_img_qc_output/BR00143980/Image.csv -> /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/whole_img_qc_output/BR00143980